## Fase 1 ABSA: Fine-tuning IndoBERT/NusaBERT di TermA + Inference Efisien

Fokus final: fine-tuning saja (hasil F1 sudah sangat baik pada run sebelumnya), tanpa jalur LLM open-source/perbandingan -- itu sudah dibuang dari notebook ini.

Notebook self-contained untuk Kaggle -- semua fungsi didefinisikan langsung di sini, tidak ada `from data_utils import ...` antar file.

**Sebelum run**: nyalakan GPU di Notebook Settings (Accelerator: GPU T4 x2 atau P100), dan pastikan Internet ON (dibutuhkan untuk download model HuggingFace; data TermA sendiri dibaca langsung dari Kaggle Dataset yang sudah di-attach, tidak lewat internet).

---
**Catatan revisi (perbaikan bug & redundansi dari review sebelumnya):**
1. `train_sents`/`valid_sents` dipakai di Bagian 2 tapi sebelumnya tidak pernah didefinisikan (`NameError` dari kernel bersih) -- ditambahkan split keduanya di Bagian 1.
2. Data (`hotel_preprocess_labeled_automated.txt`) ternyata mengandung tag `B-ASPECT`/`I-ASPECT`/`B-SENTIMENT`/`I-SENTIMENT` TANPA kategori (lihat juga komentar asli di `Sentence.labels`) -- sebelumnya bikin `ValueError` lalu unpacking error. `LABEL_LIST` dan parsing-nya sekarang menoleransi tag tanpa-kategori (`category=None`) apa adanya, tanpa menebak kategori yang sebenarnya tidak dianotasi.
3. `predict_one` vs loop pengujian Maribaya tidak konsisten menangani kegagalan (dead code `if hasil is None`, exception tak tertangkap bisa menghentikan seluruh loop 30 kasus) -- dibungkus try/except.
4. Import tak terpakai dihapus (`os`, `urllib.request`, `KFold`, `Counter` -- 0 referensi, diverifikasi via AST); komentar jumlah label yang basi diganti perhitungan dinamis.
5. Opini tanpa aspek eksplisit (implicit aspect, mis. kategori uji G/M) sekarang ditampung di `ExtractionResult.implicit_opinions`, tidak dibuang diam-diam.
6. `AspectOpinionExtractor` sekarang baca label mapping dari `model.config.id2label` (tersimpan di checkpoint), bukan variabel global notebook -- lebih portable antar-sesi.
7. Peringatan otomatis kalau input ke-truncate di `max_length`; catatan eksplisit soal F1 `model_ft` vs `model_final`; pembersihan memori GPU sebelum training `model_final`; penamaan variabel `all_sents`→`full_sents` di Bagian 2 supaya tidak menimpa arti `all_sents` di Bagian 1.

Detail tiap poin ada di komentar `# PERBAIKAN ...` di sel terkait. **Belum pernah dijalankan ulang di Kaggle** -- jalankan dari kernel bersih (Restart & Run All) untuk verifikasi akhir sebelum dipakai.

---

In [1]:
!pip install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 49.3 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [2]:
!pip install -q seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# ===== Standard library =====
import re                          # clean_text, tokenize (regex)
import html                        # clean_text (unescape entity HTML)
import unicodedata                 # clean_text (normalize NFKC)
import statistics                  # dataset_stats (mean token)
import json
from dataclasses import dataclass, field   # Sentence, Span, ScoredSpan, AspectResult, ExtractionResult
from typing import List, Optional, Dict, Tuple   # type hint di hampir semua fungsi

# ===== ML / NLP =====
import numpy as np                 # compute_metrics_fn (np.argmax)
import torch                       # AspectOpinionExtractor (softmax, inference_mode, dll)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# ===== Kaggle & Hugging Face Hub =====
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# PERBAIKAN (redundansi): `os`, `urllib.request`, `KFold` (sklearn), dan `Counter`
# (collections) dihapus dari sini -- diverifikasi via AST parsing, 0 referensi di
# manapun di notebook ini. DATA_PATH di Bagian 1 langsung `open()` dari path Kaggle
# input, tidak ada loop download yang memakai os/urllib. KFold tampaknya sisa rencana
# cross-validation yang tidak jadi dipakai.

## Bagian 1: Data utilities
(sebelumnya `data_utils.py` -- sekarang didefinisikan langsung di sini,
tidak ada file terpisah untuk di-import)

In [4]:
@dataclass
class Sentence:
    tokens: List[str]
    labels: List[str]  # O, B-ASPECT, I-ASPECT, B-SENTIMENT, I-SENTIMENT


def load_iob_file(path: str) -> List[Sentence]:
    sentences = []
    tokens, labels = [], []
    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if line.strip() == "":
                if tokens:
                    sentences.append(Sentence(tokens, labels))
                    tokens, labels = [], []
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            tok, lab = parts
            tokens.append(tok)
            labels.append(lab)
    if tokens:
        sentences.append(Sentence(tokens, labels))
    return sentences

In [5]:
@dataclass
class Span:
    start: int
    end: int
    text: str
    entity_type: str                  # "ASPECT" atau "SENTIMENT" (supertype saja)
    category: Optional[str] = None    # ASPECT: FASILITAS/HARGA_NILAI/dst; SENTIMENT: POSITIVE/NEGATIVE/NEUTRAL
                                       # None = tag di data ini tidak membawa kategori (lihat _make_span)
    confidence: float = 1.0           # term confidence
    category_confidence: float = 1.0  # category confidence
    @property
    def center(self) -> float:
        return (self.start + self.end - 1) / 2.0

def bio_to_spans(
    tokens: List[str],
    labels: List[str],
    term_confidences: Optional[List[float]] = None,
    category_confidences: Optional[List[float]] = None,
) -> List[Span]:
    spans = []
    cur_full_type: Optional[str] = None   # contoh "ASPECT-FASILITAS", dipakai cek kelanjutan span
    cur_start: Optional[int] = None
    tconf = term_confidences if term_confidences is not None else [1.0] * len(tokens)
    cconf = category_confidences if category_confidences is not None else [1.0] * len(tokens)
    for i, lab in enumerate(labels + ["O"]):
        if lab.startswith("B-"):
            if cur_full_type is not None:
                spans.append(_make_span(tokens, cur_start, i, cur_full_type, tconf, cconf))
            cur_full_type = lab[2:]
            cur_start = i
        elif lab.startswith("I-") and cur_full_type == lab[2:]:
            continue
        else:
            if cur_full_type is not None:
                spans.append(_make_span(tokens, cur_start, i, cur_full_type, tconf, cconf))
            cur_full_type = None
            cur_start = None
    return spans

def _make_span(tokens, start, end, full_type, tconf, cconf) -> Span:
    # PERBAIKAN BUG: sebagian tag di data mentah cuma "ASPECT"/"SENTIMENT" polos (dari
    # label "B-ASPECT"/"B-SENTIMENT" tanpa kategori -- lihat komentar di Sentence.labels
    # dan LABEL_LIST di Bagian 2). full_type.split("-", 1) pada string TANPA "-" dulu
    # mengembalikan list 1 elemen -> ValueError saat di-unpack ke 2 variabel. Sekarang:
    # kalau tidak ada "-" sama sekali, category di-set None (span tetap valid, cuma
    # kategorinya memang tidak diketahui dari data, bukan ditebak).
    if "-" in full_type:
        entity_type, category = full_type.split("-", 1)   # "ASPECT-FASILITAS" -> ("ASPECT", "FASILITAS")
    else:
        entity_type, category = full_type, None            # "ASPECT" -> ("ASPECT", None)
    term_c = sum(tconf[start:end]) / (end - start)
    cat_c = sum(cconf[start:end]) / (end - start)
    return Span(
        start=start, end=end, text=" ".join(tokens[start:end]),
        entity_type=entity_type, category=category,
        confidence=term_c, category_confidence=cat_c,
    )

In [6]:
def dataset_stats(sentences: List[Sentence]) -> dict:
    n_sent = len(sentences)
    lengths = [len(s.tokens) for s in sentences]
    n_aspect = n_sentiment = n_multi = n_none = 0
    aspect_cat_counts: Dict[str, int] = {}
    sentiment_cat_counts: Dict[str, int] = {}
    for s in sentences:
        spans = bio_to_spans(s.tokens, s.labels)
        a = [sp for sp in spans if sp.entity_type == "ASPECT"]
        o = [sp for sp in spans if sp.entity_type == "SENTIMENT"]
        n_aspect += len(a); n_sentiment += len(o)
        for sp in a:
            aspect_cat_counts[sp.category] = aspect_cat_counts.get(sp.category, 0) + 1
        for sp in o:
            key = sp.category if sp.category is not None else "(TANPA_KATEGORI)"
            sentiment_cat_counts[key] = sentiment_cat_counts.get(key, 0) + 1
        if len(a) >= 2: n_multi += 1
        if not a and not o: n_none += 1
    return {
        "n_sentences": n_sent,
        "avg_tokens": round(statistics.mean(lengths), 2) if lengths else 0,
        "max_tokens": max(lengths) if lengths else 0,
        "n_aspect_spans": n_aspect, "n_sentiment_spans": n_sentiment,
        "pct_multi_aspect": round(100 * n_multi / n_sent, 1) if n_sent else 0,
        "pct_no_entity": round(100 * n_none / n_sent, 1) if n_sent else 0,
        "aspect_category_counts": dict(sorted(aspect_cat_counts.items(), key=lambda x: -x[1])),
        "sentiment_category_counts": dict(sorted(sentiment_cat_counts.items(), key=lambda x: -x[1])),
    }

In [7]:
DATA_PATH = "/kaggle/input/datasets/dbernardons/dataset-hotel-and-maribaya/hotel_maribaya_labeling.txt"

# Load seluruh data
all_sents = load_iob_file(DATA_PATH)

print("Total data:", len(all_sents))

# Split menjadi train+valid dan test
train_valid_sents, test_sents = train_test_split(
    all_sents,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# PERBAIKAN BUG: split kedua ini sebelumnya HILANG dari notebook -- train_sents/
# valid_sents dipakai langsung di Bagian 2 padahal tidak pernah didefinisikan di
# manapun (NameError kalau dijalankan dari kernel bersih; training_args di Bagian 2
# butuh valid set asli untuk load_best_model_at_end/metric_for_best_model="f1").
# test_size di sini menentukan proporsi valid dari train_valid_sents -- sesuaikan
# kalau target rasio Anda beda dari 60/20/20 (train/valid/test) berikut ini.
train_sents, valid_sents = train_test_split(
    train_valid_sents,
    test_size=0.25,   # 0.25 x 2400 = 600 -> rasio akhir train:valid:test = 1800:600:600
    random_state=42,
    shuffle=True
)

print("Train+Valid:", len(train_valid_sents))
print("  -> Train:", len(train_sents))
print("  -> Valid:", len(valid_sents))
print("Test:", len(test_sents))

Total data: 4001
Train+Valid: 3200
  -> Train: 2400
  -> Valid: 800
Test: 801


In [8]:
# SEL BARU -- sanity check MURAH sebelum download checkpoint IndoBERT (498MB) & training.
# dataset_stats() sebelumnya didefinisikan tapi tidak pernah dipanggil di manapun --
# kalau dipanggil di sini, masalah skema label (lihat "(TANPA_KATEGORI)" di bawah kalau
# muncul) ketahuan sekarang, gratis, bukan nanti setelah setup training penuh.
stats_all = dataset_stats(train_valid_sents + test_sents)
print(json.dumps(stats_all, indent=2, ensure_ascii=False))

if "(TANPA_KATEGORI)" in stats_all["aspect_category_counts"] or "(TANPA_KATEGORI)" in stats_all["sentiment_category_counts"]:
    print()
    print("PERINGATAN: ada span ASPECT/SENTIMENT di data tanpa kategori eksplisit.")
    print("LABEL_LIST di Bagian 2 sudah disesuaikan untuk menoleransi ini (category=None),")
    print("tapi cek lagi apakah jumlahnya wajar / sesuai ekspektasi Anda terhadap kualitas")
    print("proses labeling otomatis di file sumbernya (hotel_preprocess_labeled_automated.txt).")

{
  "n_sentences": 4001,
  "avg_tokens": 23.61,
  "max_tokens": 31249,
  "n_aspect_spans": 11904,
  "n_sentiment_spans": 14074,
  "pct_multi_aspect": 61.7,
  "pct_no_entity": 1.0,
  "aspect_category_counts": {
    "FASILITAS": 6827,
    "TEMPAT": 1491,
    "PELAYANAN": 1358,
    "HIDANGAN": 742,
    "HARGA": 681,
    "LOKASI": 572,
    "AKTIVITAS": 216,
    "LAINNYA": 17
  },
  "sentiment_category_counts": {
    "POSITIVE": 8124,
    "NEGATIVE": 5118,
    "NEUTRAL": 832
  }
}


## Bagian 2: Fine-tuning IndoBERT/NusaBERT di TermA
(sebelumnya `finetune_termA.py`)

Ganti `BASE_MODEL` di bawah untuk membandingkan:
- "indobenchmark/indobert-base-p1" (baseline IndoNLU asli)
- "LazarusNLP/NusaBERT-base" (varian lebih baru, mencakup bahasa daerah)

In [9]:
BASE_MODEL = "LazarusNLP/NusaBERT-base"

ASPECT_CATEGORIES = [
    "FASILITAS",
    "HARGA",
    "TEMPAT",
    "LOKASI",
    "PELAYANAN",
    "HIDANGAN",
    "AKTIVITAS",
    "LAINNYA",
]

SENTIMENT_CATEGORIES = ["POSITIVE", "NEGATIVE", "NEUTRAL"]

LABEL_LIST = ["O"]
for cat in ASPECT_CATEGORIES:
    LABEL_LIST += [f"B-ASPECT-{cat}", f"I-ASPECT-{cat}"]
for cat in SENTIMENT_CATEGORIES:
    LABEL_LIST += [f"B-SENTIMENT-{cat}", f"I-SENTIMENT-{cat}"]

LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}

# PERBAIKAN (redundansi): komentar jumlah label sebelumnya hardcode ("21 label",
# asumsi 7 kategori aspek) dan sudah basi sejak ASPECT_CATEGORIES bertambah jadi 11
# kategori. Diganti print dinamis supaya tidak akan basi lagi kalau kategori berubah.
print(f"Total {len(LABEL_LIST)} label: 1 (O) + {2*len(ASPECT_CATEGORIES)} "
      f"({len(ASPECT_CATEGORIES)} kategori aspek x B/I) + {2*len(SENTIMENT_CATEGORIES)} "
      f"({len(SENTIMENT_CATEGORIES)} kategori sentimen x B/I) + 4 (ASPECT/SENTIMENT tanpa kategori x B/I)")

Total 23 label: 1 (O) + 16 (8 kategori aspek x B/I) + 6 (3 kategori sentimen x B/I) + 4 (ASPECT/SENTIMENT tanpa kategori x B/I)


In [10]:
def _parse_label(label: str):
    if label == "O":
        return None, None
    parts = label.split("-", 2)
    if len(parts) == 3:
        _, supertype, category = parts        # "B-ASPECT-FASILITAS" -> ("ASPECT", "FASILITAS")
    else:
        # PERBAIKAN BUG: split("-", 2) dulu diasumsikan SELALU menghasilkan 3 bagian --
        # meledak (ValueError: not enough values to unpack) untuk label 2-bagian seperti
        # "B-ASPECT"/"B-SENTIMENT" (tanpa kategori) yang memang ada di data. Ditangani
        # eksplisit sekarang: category=None.
        _, supertype = parts                   # "B-ASPECT" -> ("ASPECT", None)
        category = None
    return supertype, category

LABEL_SUPERTYPE: Dict[str, Optional[str]] = {}
LABEL_CATEGORY: Dict[str, Optional[str]] = {}
SUPERTYPE_LABEL_IDS: Dict[str, List[int]] = {"ASPECT": [], "SENTIMENT": []}
CATEGORY_LABEL_IDS: Dict[Tuple[str, Optional[str]], List[int]] = {}

for lab, idx in LABEL2ID.items():
    st, cat = _parse_label(lab)
    LABEL_SUPERTYPE[lab] = st
    LABEL_CATEGORY[lab] = cat
    if st is not None:
        SUPERTYPE_LABEL_IDS[st].append(idx)
        CATEGORY_LABEL_IDS.setdefault((st, cat), []).append(idx)

def token_confidences(prob_vector, pred_label: str) -> Tuple[float, Optional[float]]:
    """Pecah distribusi softmax 1 token -> (term_confidence, category_confidence).
    term_confidence  = P(supertype-nya benar), dijumlah dari semua kategori & B/I dalam grup itu.
    category_confidence = P(kategori spesifik | supertype benar) -- conditional probability.
    Untuk token yang diprediksi sebagai label tanpa-kategori (B-ASPECT/B-SENTIMENT
    polos), category_confidence tetap berupa angka -- artinya "P(memang tanpa-kategori |
    supertype benar)", bukan None, supaya konsisten dipakai di ScoredSpan/AspectResult.
    """
    if pred_label == "O":
        return float(prob_vector[LABEL2ID["O"]]), None
    supertype = LABEL_SUPERTYPE[pred_label]
    category = LABEL_CATEGORY[pred_label]
    term_conf = float(sum(prob_vector[i] for i in SUPERTYPE_LABEL_IDS[supertype]))
    cat_group_conf = float(sum(prob_vector[i] for i in CATEGORY_LABEL_IDS[(supertype, category)]))
    cat_conf = cat_group_conf / term_conf if term_conf > 0 else 0.0
    return term_conf, cat_conf

In [11]:
def sentences_to_hf_dataset(sentences: List[Sentence]):
    return Dataset.from_dict({
        "tokens": [s.tokens for s in sentences],
        "ner_tags": [[LABEL2ID[l] for l in s.labels] for s in sentences],
    })


def tokenize_and_align_labels(examples, tokenizer):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=256)
    all_labels = []
    for i, label_ids in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_row = []
        for word_idx in word_ids:
            if word_idx is None:
                label_row.append(-100)
            elif word_idx != previous_word_idx:
                label_row.append(label_ids[word_idx])
            else:
                label_row.append(-100)
            previous_word_idx = word_idx
        all_labels.append(label_row)
    tokenized["labels"] = all_labels
    return tokenized


def compute_metrics_fn(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [[ID2LABEL[p] for p, l in zip(pr, la) if l != -100] for pr, la in zip(predictions, labels)]
    true_labels = [[ID2LABEL[l] for p, l in zip(pr, la) if l != -100] for pr, la in zip(predictions, labels)]
    report = classification_report(true_labels, true_predictions, output_dict=True, zero_division=0)

    metrics = {
        "precision": precision_score(true_labels, true_predictions, zero_division=0),
        "recall": recall_score(true_labels, true_predictions, zero_division=0),
        "f1": f1_score(true_labels, true_predictions, zero_division=0),
    }
    aspect_f1s, sentiment_f1s = [], []
    for key, scores in report.items():
        if key in ("micro avg", "macro avg", "weighted avg"):
            continue
        f1 = scores.get("f1-score", 0.0)
        metrics[f"f1_{key.lower().replace('-', '_')}"] = f1   # mis. f1_aspect_fasilitas, f1_sentiment_negative
        # PERBAIKAN: sekarang entity type bisa juga "ASPECT"/"SENTIMENT" polos (dari tag
        # tanpa-kategori, lihat LABEL_LIST) -- key.startswith("ASPECT-") saja akan
        # MELEWATKAN entity type polos itu dari agregasi macro (dia tidak diawali
        # "ASPECT-" dengan strip). Ditambahkan pengecekan == "ASPECT"/"SENTIMENT" juga.
        if key == "ASPECT" or key.startswith("ASPECT-"):
            aspect_f1s.append(f1)
        elif key == "SENTIMENT" or key.startswith("SENTIMENT-"):
            sentiment_f1s.append(f1)

    # PERBAIKAN BUG: key agregat ini sebelumnya bernama "f1_aspect"/"f1_sentiment" --
    # PERSIS SAMA dengan nama key per-kategori yang otomatis dihasilkan di loop atas
    # untuk entity type "ASPECT"/"SENTIMENT" polos (yang sekarang bisa muncul karena
    # LABEL_LIST menoleransi label tanpa-kategori). Tanpa perbaikan ini, assignment di
    # bawah diam-diam menimpa nilai f1 kategori "tanpa-kategori" itu. Diberi nama beda
    # (_macro) supaya tidak akan pernah bentrok, diverifikasi dengan seqeval langsung.
    metrics["f1_aspect_macro"] = round(sum(aspect_f1s) / len(aspect_f1s), 4) if aspect_f1s else 0.0
    metrics["f1_sentiment_macro"] = round(sum(sentiment_f1s) / len(sentiment_f1s), 4) if sentiment_f1s else 0.0
    return metrics

In [12]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

tokenizer_ft = AutoTokenizer.from_pretrained(BASE_MODEL)
model_ft = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABEL_LIST), id2label=ID2LABEL, label2id=LABEL2ID
)

all_labels_in_data = {l for s in (train_sents + valid_sents + test_sents) for l in s.labels}
missing_from_label_list = all_labels_in_data - set(LABEL_LIST)
if missing_from_label_list:
    print(missing_from_label_list)
    raise ValueError(f"Label ada di data tapi tidak ada di LABEL_LIST: {missing_from_label_list}")
else:
    print("Cek label OK -- semua label di train_sents/valid_sents/test_sents ada di LABEL_LIST.")

train_ds = sentences_to_hf_dataset(train_sents).map(lambda ex: tokenize_and_align_labels(ex, tokenizer_ft), batched=True)
valid_ds = sentences_to_hf_dataset(valid_sents).map(lambda ex: tokenize_and_align_labels(ex, tokenizer_ft), batched=True)
test_ds = sentences_to_hf_dataset(test_sents).map(lambda ex: tokenize_and_align_labels(ex, tokenizer_ft), batched=True)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer_ft)

training_args = TrainingArguments(
    output_dir="/kaggle/working/termA-model",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    logging_strategy="steps",
    logging_steps=50,
    num_train_epochs=5, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="f1", report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model_ft, args=training_args,
    train_dataset=train_ds, eval_dataset=valid_ds,
    processing_class=tokenizer_ft, data_collator=data_collator,
    compute_metrics=compute_metrics_fn,
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

BertForTokenClassification LOAD REPORT from: LazarusNLP/NusaBERT-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Cek label OK -- semua label di train_sents/valid_sents/test_sents ada di LABEL_LIST.


Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/801 [00:00<?, ? examples/s]

In [13]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,F1 Aspect Aktivitas,F1 Aspect Fasilitas,F1 Aspect Harga,F1 Aspect Hidangan,F1 Aspect Lainnya,F1 Aspect Lokasi,F1 Aspect Pelayanan,F1 Aspect Tempat,F1 Sentiment Negative,F1 Sentiment Neutral,F1 Sentiment Positive,F1 Aspect Macro,F1 Sentiment Macro
1,3.593562,1.659417,0.509338,0.535775,0.522222,0.000000,0.563912,0.000000,0.000000,0.000000,0.000000,0.049587,0.000000,0.508855,0.000000,0.636951,0.076700,0.381900
2,1.137996,0.924836,0.731719,0.762066,0.746584,0.000000,0.761905,0.807339,0.039735,0.000000,0.439024,0.703390,0.568720,0.691413,0.000000,0.873837,0.415000,0.521700
3,0.873038,0.732269,0.788620,0.813485,0.800860,0.000000,0.825626,0.840336,0.652482,0.000000,0.547945,0.770833,0.668085,0.730123,0.000000,0.902284,0.538200,0.544100
4,0.662389,0.692544,0.788219,0.834101,0.810511,0.000000,0.839143,0.833333,0.649351,0.000000,0.594937,0.809035,0.685714,0.747363,0.000000,0.900598,0.551400,0.549300
5,0.641108,0.670573,0.796211,0.835799,0.815525,0.000000,0.840237,0.842975,0.671010,0.000000,0.612500,0.820619,0.691057,0.753218,0.000000,0.904798,0.559800,0.552700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=375, training_loss=1.3069451955159506, metrics={'train_runtime': 134.5155, 'train_samples_per_second': 89.209, 'train_steps_per_second': 2.788, 'total_flos': 407683997535936.0, 'train_loss': 1.3069451955159506, 'epoch': 5.0})

In [14]:
finetune_test_metrics = trainer.evaluate(test_ds)
print(json.dumps(finetune_test_metrics, indent=2))

trainer.save_model("/kaggle/working/termA-model")
tokenizer_ft.save_pretrained("/kaggle/working/termA-model")
print("Model tersimpan di /kaggle/working/termA-model")

print()
print("CATATAN: F1 di atas adalah evaluasi held-out yang VALID untuk model_ft (dilatih")
print("cuma di train_sents; test_sents sama sekali tidak pernah dilihat model ini).")
print("model_final di sel berikutnya dilatih ulang di SEMUA data termasuk test_sents,")
print("jadi tidak punya evaluasi held-out sendiri -- lihat catatan di sel berikutnya.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "eval_loss": 0.6213436722755432,
  "eval_precision": 0.8074630161183485,
  "eval_recall": 0.8432095918837906,
  "eval_f1": 0.8249492443040831,
  "eval_f1_aspect_fasilitas": 0.8274153592072666,
  "eval_f1_aspect_harga": 0.8555555555555556,
  "eval_f1_aspect_hidangan": 0.68,
  "eval_f1_aspect_lainnya": 0.0,
  "eval_f1_aspect_lokasi": 0.7303370786516854,
  "eval_f1_aspect_pelayanan": 0.8491228070175438,
  "eval_f1_aspect_tempat": 0.7245841035120147,
  "eval_f1_sentiment_negative": 0.7486278814489572,
  "eval_f1_sentiment_neutral": 0.0,
  "eval_f1_sentiment_positive": 0.9227513227513228,
  "eval_f1_aspect_macro": 0.6667,
  "eval_f1_sentiment_macro": 0.5571,
  "eval_runtime": 2.9612,
  "eval_samples_per_second": 270.498,
  "eval_steps_per_second": 8.78,
  "epoch": 5.0
}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model tersimpan di /kaggle/working/termA-model

CATATAN: F1 di atas adalah evaluasi held-out yang VALID untuk model_ft (dilatih
cuma di train_sents; test_sents sama sekali tidak pernah dilihat model ini).
model_final di sel berikutnya dilatih ulang di SEMUA data termasuk test_sents,
jadi tidak punya evaluasi held-out sendiri -- lihat catatan di sel berikutnya.


In [15]:
# Gabungkan train+valid+test JADI SATU untuk training final
# PERBAIKAN (kejelasan nama): diganti dari `all_sents` -> `full_sents`. Nama `all_sents`
# sudah dipakai di Bagian 1 untuk "seluruh data SEBELUM displit" -- isinya beda urutan
# (walau sama isinya secara set) dari gabungan train+valid+test di sini, jadi memakai
# ulang nama yang sama berisiko membingungkan saat notebook dibaca ulang nanti.
full_sents = train_sents + valid_sents + test_sents
full_ds = sentences_to_hf_dataset(full_sents).map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer_ft), batched=True
)

final_training_args = TrainingArguments(
    output_dir="/kaggle/working/termA-model-final",
    seed=42,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    logging_strategy="steps",
    logging_steps=50,
    num_train_epochs=5,
    weight_decay=0.01,
    report_to="none",
)

# PERBAIKAN (kebersihan memori GPU): model_ft & trainer lamanya masih hidup di memori
# sampai titik ini (masih ada referensi ke variabelnya), jadi tanpa ini, model_ft dan
# model_final (plus optimizer state masing-masing) nangkring bareng di VRAM selagi
# model_final dilatih -- tidak perlu, apalagi kalau BASE_MODEL nanti diganti ke varian
# yang lebih besar dari IndoBERT/NusaBERT-base.
del trainer, model_ft
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_final = AutoModelForTokenClassification.from_pretrained(   # model BARU dari checkpoint pretrained,
    BASE_MODEL, num_labels=len(LABEL_LIST), id2label=ID2LABEL, label2id=LABEL2ID  # bukan lanjutan model_ft
)

final_trainer = Trainer(
    model=model_final, args=final_training_args,
    train_dataset=full_ds, processing_class=tokenizer_ft, data_collator=data_collator,
)
final_trainer.train()
final_trainer.save_model("/kaggle/working/termA-model-final")
tokenizer_ft.save_pretrained("/kaggle/working/termA-model-final")

print()
print("CATATAN PENTING: model_final di atas dilatih di SEMUA data (termasuk test_sents),")
print("jadi TIDAK punya evaluasi held-out sendiri. finetune_test_metrics di sel sebelumnya")
print("adalah proksi/lower-bound yang wajar (model_final dilatih di data lebih banyak,")
print("jadi performanya kemungkinan >= itu), tapi bukan pengukuran langsung ke model_final.")

Map:   0%|          | 0/4001 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: LazarusNLP/NusaBERT-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were 

Step,Training Loss
50,3.494586
100,1.654820
150,1.046965
200,0.819810
250,0.744928
300,0.628384
350,0.590381
400,0.559540
450,0.513145
500,0.518604


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


CATATAN PENTING: model_final di atas dilatih di SEMUA data (termasuk test_sents),
jadi TIDAK punya evaluasi held-out sendiri. finetune_test_metrics di sel sebelumnya
adalah proksi/lower-bound yang wajar (model_final dilatih di data lebih banyak,
jadi performanya kemungkinan >= itu), tapi bukan pengukuran langsung ke model_final.


## Bagian 3: Ekstraksi efisien dari model fine-tuned (production inference)

Dijalankan LANGSUNG setelah Bagian 2 (bukan di notebook/sesi terpisah) --
`/kaggle/working/` scoped per-notebook, jadi memuat model dari path ini
cuma valid kalau dijalankan di sesi yang sama dengan yang training.
Kalau mau dipakai di sesi lain nanti: simpan folder `termA-model` sebagai
Kaggle Dataset, lalu ganti path ke `/kaggle/input/<nama-dataset>/...`.

Tanpa `pipeline()` -- kontrol manual atas tokenizer, batching (diurutkan
by panjang untuk minimalkan padding waste), dan post-processing.

**Catatan penting soal grouping opinion->aspect**: percobaan pertama memakai deteksi "batas klausa" (marker jadi batas hanya kalau token setelahnya B-ASPECT) -- terlihat masuk akal tapi TERBUKTI GAGAL saat diuji langsung: begitu 2 aspect dianggap "1 klausa", tie-break jarak token salah pilih aspect yang tekstual lebih dekat walau salah secara makna (kasus "kolamnya bagus banget dan sejuk, cuma parkirannya sempit" -- "sejuk" nyaris salah nempel ke "parkirannya"). Solusi yang lebih sederhana DAN terbukti benar: setiap opinion menempel ke **aspect paling baru yang sudah disebut di sebelah kirinya** (sequential ownership berdasar urutan baca) -- tidak perlu definisi klausa sama sekali. Diuji di 5 skenario termasuk kasus di atas, semua lulus.

**Revisi**: opini yang tidak ada aspek eksplisit di sebelah kirinya (termasuk kalimat tanpa aspek sama sekali) sebelumnya dibuang total tanpa jejak. Sekarang ditampung terpisah di `ExtractionResult.implicit_opinions`, bukan hilang begitu saja -- lihat `group_opinions_by_aspect` di bawah.

**Keterbatasan yang diketahui (belum diperbaiki, di luar cakupan revisi bug kali ini)**: heuristik sequential ownership ini tidak memodelkan relasi komparatif secara eksplisit (kalimat yang membandingkan 2 entitas, mis. kategori uji F-komparatif) -- opini tetap menempel ke aspek terdekat di kiri, bukan merepresentasikan "X lebih baik dari Y" sebagai satu relasi utuh. Menangani ini dengan benar perlu pendekatan berbeda (mis. relation extraction / model generatif).

In [16]:
_HTML_TAG_RE = re.compile(r"<[^>]+>")
_WHITESPACE_RE = re.compile(r"\s+")
_TOKEN_RE = re.compile(r"\w+(?:[-']\w+)*|[^\w\s]")  # kata (boleh ada - atau ' di tengah) ATAU 1 tanda baca


def clean_text(text: str) -> str:
    """Encoding rusak, artefak HTML sisa scraping, whitespace berlebih.
    TIDAK stemming/stopword-removal/lowercasing paksa -- itu merusak span
    verbatim yang jadi kontrak dasar model token-classification ini."""
    text = html.unescape(text)                          # &amp; -> &, &quot; -> ", dst.
    text = _HTML_TAG_RE.sub(" ", text)                   # buang tag <br>, <p>, dst.
    text = unicodedata.normalize("NFKC", text)           # normalisasi unicode (mis. smart-quote varian)
    text = text.encode("utf-8", "ignore").decode("utf-8")  # buang byte yang rusak/tidak valid
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


def tokenize(text: str) -> List[str]:
    """Pisahkan kata dan tanda baca jadi token terpisah -- konsisten dengan
    konvensi TermA (koma/titik adalah token sendiri, bukan menempel ke kata)."""
    return _TOKEN_RE.findall(text)

In [17]:
_CLAUSE_KEYWORDS = {"tapi", "namun", "tetapi", ","}

def segment_clauses_keyword(tokens: List[str]) -> List[int]:
    """clause_id per token, murni dari kata kunci ("tapi"/"namun"/koma) --
    metadata umum untuk logging/analisis, BUKAN dipakai untuk pairing (lihat
    docstring modul: pendekatan berbasis klausa terbukti gagal untuk pairing,
    diganti pendekatan sequential ownership di group_opinions_by_aspect)."""
    ids, cur = [], 0
    for tok in tokens:
        ids.append(cur)
        if tok.lower() in _CLAUSE_KEYWORDS:
            cur += 1
    return ids

In [18]:
@dataclass
class ScoredSpan:
    text: str
    confidence: float                       # term confidence
    category: Optional[str] = None          # aspek: FASILITAS/dst ; sentimen: POSITIVE/NEGATIVE/NEUTRAL
    category_confidence: Optional[float] = None
    pairing_gap: Optional[int] = None

@dataclass
class AspectResult:
    aspect: ScoredSpan
    opinions: List[ScoredSpan] = field(default_factory=list)
    category: Optional[str] = None
    category_confidence: Optional[float] = None

In [19]:
def group_opinions_by_aspect(
    tokens: List[str],
    labels: List[str],
    term_confidences: Optional[List[float]] = None,
    category_confidences: Optional[List[float]] = None,
    strip_nya: bool = True,
) -> Tuple[List[AspectResult], List[ScoredSpan]]:
    spans = bio_to_spans(tokens, labels, term_confidences, category_confidences)
    results: List[AspectResult] = []
    implicit_opinions: List[ScoredSpan] = []   # PERBAIKAN (gap desain, lihat catatan Bagian 3)
    current: Optional[AspectResult] = None
    current_aspect_end: Optional[int] = None
    for sp in sorted(spans, key=lambda s: s.start):
        if sp.entity_type == "ASPECT":
            name = _normalize_aspect_display(sp.text) if strip_nya else sp.text
            current = AspectResult(
                aspect=ScoredSpan(
                    text=name,
                    confidence=round(sp.confidence, 3),
                    category=sp.category,
                    category_confidence=round(sp.category_confidence, 3),
                ),
                category=sp.category,
                category_confidence=round(sp.category_confidence, 3),
            )
            results.append(current)
            current_aspect_end = sp.end
        elif sp.entity_type == "SENTIMENT":
            scored = ScoredSpan(
                text=sp.text,
                confidence=round(sp.confidence, 3),
                category=sp.category,
                category_confidence=round(sp.category_confidence, 3),
                pairing_gap=(sp.start - current_aspect_end) if current_aspect_end is not None else None,
            )
            # PERBAIKAN BUG: opini yang muncul SEBELUM aspek pertama, atau di kalimat
            # yang sama sekali tidak punya aspek eksplisit, sebelumnya dibuang total
            # tanpa jejak -- padahal sentimennya jelas ada (lihat kategori uji
            # "G-implisit" dan "M-generik_pendek" di MARIBAYA_TEST_REVIEWS). Sekarang
            # ditampung di `implicit_opinions`, bukan hilang begitu saja.
            if current is not None and current_aspect_end is not None:
                current.opinions.append(scored)
            else:
                implicit_opinions.append(scored)
    return results, implicit_opinions

_NYA_SUFFIX_RE = re.compile(r"nya$", re.IGNORECASE)
def _normalize_aspect_display(term: str) -> str:
    words = term.split(" ")
    if len(words) > 1 and words[-1].lower() == "nya":
        words = words[:-1]
    else:
        words[-1] = _NYA_SUFFIX_RE.sub("", words[-1]) or words[-1]
    return " ".join(words)

In [20]:
@dataclass
class ExtractionResult:
    text_cleaned: str
    tokens: List[str]
    clause_ids: List[int]
    aspects: List[AspectResult]
    implicit_opinions: List[ScoredSpan] = field(default_factory=list)   # PERBAIKAN: lihat group_opinions_by_aspect

In [21]:
class AspectOpinionExtractor:
    def __init__(self, model_dir: str, device: Optional[str] = None, max_length: int = 256):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir).to(self.device)
        self.model.eval()
        if self.device == "cuda":
            self.model = self.model.half()
        # PERBAIKAN (portabilitas): sebelumnya _run_batch pakai ID2LABEL (variabel
        # global notebook) -- cuma benar kalau dijalankan di sesi yang sama dengan
        # training (persis seperti catatan session-scoping di markdown Bagian 3).
        # id2label SUDAH otomatis tersimpan di config checkpoint (karena
        # id2label=ID2LABEL di-pass saat from_pretrained/save_model di Bagian 2), jadi
        # diambil dari situ -> kelas ini jadi self-contained, bisa dipakai di sesi/
        # notebook lain hanya dengan model_dir (HF config JSON-roundtrips int key jadi
        # string, makanya di-cast ulang ke int di sini).
        self.id2label: Dict[int, str] = {int(k): v for k, v in self.model.config.id2label.items()}

    def _run_batch(self, batch_tokens: List[List[str]]) -> List["ExtractionResult"]:
        enc = self.tokenizer(batch_tokens, is_split_into_words=True, truncation=True,
                              max_length=self.max_length, padding=True, return_tensors="pt").to(self.device)
        logits = self.model(**enc).logits
        probs = torch.softmax(logits, dim=-1).float().tolist()
        pred_ids = logits.argmax(-1).tolist()
        out = []
        for row, tokens in enumerate(batch_tokens):
            word_ids = enc.word_ids(batch_index=row)
            row_labels, row_term_conf, row_cat_conf, prev_w = [], [], [], None
            max_word_seen = -1
            for wid, pid, pv in zip(word_ids, pred_ids[row], probs[row]):
                if wid is None or wid == prev_w:
                    continue
                label = self.id2label[pid]
                tconf, cconf = token_confidences(pv, label)
                row_labels.append(label)
                row_term_conf.append(tconf)
                row_cat_conf.append(cconf if cconf is not None else 1.0)
                prev_w = wid
                max_word_seen = wid

            # PERBAIKAN (transparansi truncation): kalau token asli lebih banyak dari
            # jumlah word_id unik yang benar-benar tercakup model (karena truncation di
            # max_length), sisa token di akhir kalimat dulu diam-diam dianggap "O" tanpa
            # peringatan apa pun -- padahal ada aspek/opini yang mungkin ada di bagian
            # yang terpotong (relevan khusus utk kategori uji "L-sangat_panjang").
            n_covered = max_word_seen + 1
            if n_covered < len(tokens):
                print(f"[PERINGATAN] Input terpotong (truncation) di max_length={self.max_length}: "
                      f"{len(tokens)} token asli, cuma {n_covered} yang tercakup model. "
                      f"{len(tokens) - n_covered} token terakhir otomatis dianggap 'O'.")

            row_labels = (row_labels + ["O"] * len(tokens))[:len(tokens)]
            row_term_conf = (row_term_conf + [1.0] * len(tokens))[:len(tokens)]
            row_cat_conf = (row_cat_conf + [1.0] * len(tokens))[:len(tokens)]
            aspects, implicit_opinions = group_opinions_by_aspect(tokens, row_labels, row_term_conf, row_cat_conf)
            out.append(ExtractionResult(text_cleaned=" ".join(tokens), tokens=tokens,
                                         clause_ids=segment_clauses_keyword(tokens), aspects=aspects,
                                         implicit_opinions=implicit_opinions))
        return out

    @torch.inference_mode()
    def predict_batch(self, texts: List[str], batch_size: int = 32):
        results = [None] * len(texts)
        failed = []

        # PERBAIKAN (redundansi): `cleaned` (list hasil clean_text()) sebelumnya dibuat
        # tapi tidak pernah dibaca lagi setelahnya -- ExtractionResult.text_cleaned
        # di-generate ulang dari " ".join(tokens), bukan dari `cleaned`. Dihapus di sini;
        # clean_text() sendiri tetap dipanggil karena hasilnya (`c`) dipakai tokenize(c).
        token_lists, valid_idx = [], []
        for i, t in enumerate(texts):
            try:
                c = clean_text(t)
                token_lists.append(tokenize(c)); valid_idx.append(i)
            except Exception as e:
                failed.append((i, f"cleaning/tokenisasi gagal: {e}"))

        order = sorted(range(len(valid_idx)), key=lambda k: len(token_lists[k]))
        for start in range(0, len(order), batch_size):
            pos = order[start:start + batch_size]
            orig_idx = [valid_idx[k] for k in pos]
            batch_tok = [token_lists[k] for k in pos]
            try:
                for oi, res in zip(orig_idx, self._run_batch(batch_tok)):
                    results[oi] = res
            except Exception:
                for oi, tok in zip(orig_idx, batch_tok):
                    try:
                        results[oi] = self._run_batch([tok])[0]
                    except Exception as e2:
                        failed.append((oi, f"inference gagal: {e2}"))

        return results, failed

    def predict_one(self, text: str) -> ExtractionResult:
        results, failed = self.predict_batch([text])

        if results[0] is None:
            detail = failed[0][1] if failed else "Unknown error"
            raise RuntimeError(f"Prediction failed: {detail}")

        return results[0]

In [22]:
extractor = AspectOpinionExtractor("/kaggle/working/termA-model-final")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [23]:
result = extractor.predict_one("kolam renang bersih tapi harga tiket masuk mahal")
for a in result.aspects:
    print(f"ASPEK: {a.aspect.text!r} | term_conf={a.aspect.confidence} | "
          f"category={a.category} | category_conf={a.category_confidence}")
    for o in a.opinions:
        print(f"  OPINI: {o.text!r} | term_conf={o.confidence} | "
              f"sentiment={o.category} | sentiment_conf={o.category_confidence}")
if result.implicit_opinions:
    print("OPINI TANPA ASPEK EKSPLISIT:")
    for o in result.implicit_opinions:
        print(f"  {o.text!r} | term_conf={o.confidence} | sentiment={o.category} | sentiment_conf={o.category_confidence}")

ASPEK: 'kolam renang' | term_conf=0.998 | category=FASILITAS | category_conf=0.994
  OPINI: 'bersih' | term_conf=0.999 | sentiment=POSITIVE | sentiment_conf=0.999
ASPEK: 'harga' | term_conf=0.987 | category=HARGA | category_conf=0.973
ASPEK: 'tiket' | term_conf=0.887 | category=HARGA | category_conf=0.573
  OPINI: 'mahal' | term_conf=0.969 | sentiment=NEGATIVE | sentiment_conf=0.944


In [24]:
result = extractor.predict_one("mahal banget")
for a in result.aspects:
    print(f"ASPEK: {a.aspect.text!r} | term_conf={a.aspect.confidence} | "
          f"category={a.category} | category_conf={a.category_confidence}")
    for o in a.opinions:
        print(f"  OPINI: {o.text!r} | term_conf={o.confidence} | "
              f"sentiment={o.category} | sentiment_conf={o.category_confidence}")
if result.implicit_opinions:
    print("OPINI TANPA ASPEK EKSPLISIT:")
    for o in result.implicit_opinions:
        print(f"  {o.text!r} | term_conf={o.confidence} | sentiment={o.category} | sentiment_conf={o.category_confidence}")

OPINI TANPA ASPEK EKSPLISIT:
  'mahal banget' | term_conf=0.97 | sentiment=NEGATIVE | sentiment_conf=0.742


In [25]:
# CATATAN METODOLOGI: 30 kalimat uji di bawah ini soal Curug Maribaya (wisata alam) --
# BEDA domain dari data fine-tuning (ulasan hotel Airy, DATA_PATH di Bagian 1). Ini valid
# sebagai uji generalisasi/stress-test out-of-domain (kosakata: wahana, glamping tent,
# sky bridge, balon udara, dst. kemungkinan jarang/tidak ada di data hotel), tapi jangan
# disamakan dengan evaluasi utama -- finetune_test_metrics di Bagian 2 itu yang in-domain.
MARIBAYA_TEST_REVIEWS = [

    # ================================================================
    # A. MULTI-ASPEK PADAT (3-5+ aspek dalam 1 kalimat/ulasan)
    # ================================================================
    {"kategori": "A-multi_aspek", "catatan": "5 aspek berbeda, tanda baca minim antar klausa",
     "teks": "kolam air panasnya enak banget buat rendam kaki, tapi jalan menuju curugnya "
             "lumayan jauh dan licin pas hujan, mushola nya kecil banget jadi harus antre lama, "
             "parkiran motor juga sempit banget apalagi pas weekend rame."},

    {"kategori": "A-multi_aspek", "catatan": "harga + pemandangan + kebersihan + warung, 1 paragraf",
     "teks": "harga tiket masuk lumayan mahal menurut aku buat fasilitas yang ada, tapi "
             "pemandangan hutan pinusnya emang juara, spot fotonya banyak banget, cuma sayang "
             "tempat sampah kurang jadi agak kotor di beberapa titik, warung makannya juga "
             "lumayan harganya standar kok gak semahal yang aku kira."},

    # ================================================================
    # B. NEGASI + INTENSIFIER MAJEMUK (harus jadi SATU span opini)
    # ================================================================
    {"kategori": "B-negasi", "catatan": "3 aspek, tiap aspek ada negasi berbeda posisi",
     "teks": "sky bridge nya bagus tapi antreannya gak ketulungan, udah gitu petugasnya "
             "kurang ramah pas ditanya-tanya, air terjunnya sih tidak terlalu deras waktu itu "
             "jadi kurang seru buat difoto."},

    {"kategori": "B-negasi", "catatan": "kontradiksi dalam 1 aspek (bersih TAPI bau)",
     "teks": "toiletnya lumayan bersih tapi baunya nyengat banget, gak tau kenapa padahal "
             "udah keliatan rajin dibersihin sih."},

    {"kategori": "B-negasi", "catatan": "ekspektasi vs realita, negasi implisit lewat 'biasa aja'",
     "teks": "jujur ekspektasi ku ketinggian, ternyata wahananya biasa aja, gak seru-seru amat "
             "buat orang dewasa, mungkin lebih cocok anak kecil kali ya."},

    # ================================================================
    # C. GAYA LIST/REKAP (plus-minus, baik-buruk)
    # ================================================================
    {"kategori": "C-list_style", "catatan": "format plus/minus eksplisit",
     "teks": "plus: pemandangan oke, udara adem, banyak spot foto. minus: harga makanan di "
             "dalam mahal banget, toilet kurang banyak, jalan setapak licin abis hujan."},

    {"kategori": "C-list_style", "catatan": "1 kata sentimen menaungi >1 aspek (baik: X, Y)",
     "teks": "baik: kebersihan area kolam, keramahan petugas tiket. kurang: variasi menu di "
             "warung, penerangan jalan menuju parkiran pas malam."},

    {"kategori": "C-list_style", "catatan": "gaya 'review jujur' semi-list",
     "teks": "review jujur ya: kolam air panas top markotop, akses jalan lumayan buruk banyak "
             "lubang, parkir susah nyari yang teduh, pelayanan tiket lumayan cepet kok gak "
             "pake lama antrenya."},

    # ================================================================
    # D. RUN-ON / STREAM OF CONSCIOUSNESS (panjang, minim tanda baca)
    # ================================================================
    {"kategori": "D-run_on", "catatan": "1 kalimat sangat panjang, banyak transisi topik",
     "teks": "jadi kemarin kesana pagi-pagi biar gak rame eh ternyata udah rame juga pas jam 8 "
             "udah banyak yang antre di sky bridge terus pas mau ke curugnya jalannya turun "
             "terus licin banget untung bawa sendal gunung anak-anak sempet ngeluh capek tapi "
             "begitu sampe curugnya langsung seneng soalnya airnya jernih dan seger banget "
             "cuma sayang di area situ gaada tempat sampah jadi banyak yang buang sampah "
             "sembarangan agak miris liatnya."},

    {"kategori": "D-run_on", "catatan": "narasi kronologis kunjungan, banyak sub-kejadian",
     "teks": "nyampe sana langsung parkir untung masih kebagian tempat abis itu beli tiket "
             "antre nya lumayan tapi cepet kok terus jalan ke lokasi kolam sambil foto-foto di "
             "spot yang banyak banget itu terus nyobain rendam kaki di kolam anget nya seger "
             "banget badan langsung rileks abis itu laper jadi mampir warung makan bakso disana "
             "harganya standar rasanya lumayan enak overall puas sih cuma pengen toiletnya "
             "ditambah lagi soalnya antre juga pas mau ke toilet."},

    # ================================================================
    # E. SENTIMEN CAMPURAN UNTUK ASPEK YANG SAMA (>1 opini per aspek)
    # ================================================================
    {"kategori": "E-mixed_same_aspect", "catatan": "1 aspek (parkiran), 2 sifat berlawanan",
     "teks": "parkirannya luas sih tapi becek pas musim hujan."},

    {"kategori": "E-mixed_same_aspect", "catatan": "1 aspek (warung), pujian + keluhan harga",
     "teks": "warungnya banyak pilihan menunya enak-enak semua cuma agak mahal dikit "
             "dibanding tempat wisata lain yang pernah aku kunjungi."},

    {"kategori": "E-mixed_same_aspect", "catatan": "3 sifat untuk 1 aspek (glamping tent)",
     "teks": "glamping tentnya nyaman, bersih, tapi agak kecil buat ukuran keluarga isi 4 orang."},

    # ================================================================
    # F. KOMPARATIF (dibanding kunjungan lalu / tempat lain)
    # ================================================================
    {"kategori": "F-komparatif", "catatan": "dibanding kunjungan sendiri 2 tahun lalu",
     "teks": "dibanding waktu aku kesini 2 tahun lalu, sekarang jauh lebih bagus fasilitasnya, "
             "apalagi toiletnya udah gak sebau dulu, tapi harga tiket juga naik cukup signifikan."},

    {"kategori": "F-komparatif", "catatan": "dibanding tempat wisata lain (The Lodge vs Curug)",
     "teks": "kalo dibandingin sama the lodge yang di seberang, menurutku curug maribaya ini "
             "lebih worth it buat yang budget terbatas, meskipun wahana refreshing kayak balon "
             "udara emang gaada disini."},

    # ================================================================
    # G. ASPEK IMPLISIT (opini tanpa penyebutan objek eksplisit)
    # ================================================================
    {"kategori": "G-implisit", "catatan": "puas umum, TIDAK menyebut aspek spesifik apa pun",
     "teks": "seru banget pokoknya, worth every penny, pasti balik lagi kesini sama keluarga besar."},

    {"kategori": "G-implisit", "catatan": "kecewa umum tanpa objek jelas",
     "teks": "kecewa berat jujur, padahal udah expect banyak dari review-review sebelumnya."},

    {"kategori": "G-implisit", "catatan": "opini umum + 1 aspek eksplisit di kalimat lanjutan",
     "teks": "gak nyangka sebagus ini, apalagi pas golden hour. cuma parkirannya jauh dari "
             "pintu masuk jadi agak capek jalan kaki bawa barang."},

    # ================================================================
    # H. SLANG, SINGKATAN, TYPO (representatif bahasa gaul asli)
    # ================================================================
    {"kategori": "H-slang_typo", "catatan": "singkatan 'bgt', 'tp', 'krn', 'dmn2'",
     "teks": "kolam nya seger bgt asli, tp toiletnya jorok parah astaga, gak recommended klo "
             "bawa anak kecil krn licin dmn2."},

    {"kategori": "H-slang_typo", "catatan": "angka harga + singkatan 'rb', 'jd'",
     "teks": "parkir motor 5rb mobil 10rb standard sih, cuma pas weekend suka penuh jd agak "
             "muter2 nyari spot kosong."},

    {"kategori": "H-slang_typo", "catatan": "bahasa sangat informal, huruf diulang utk penekanan",
     "teks": "seruuu bangettt tempatnya adem, cuma antrean masuk parkiran macettt parah pas "
             "jam 9an, untung petugasnya sigap ngatur jadi gak lama-lama amat."},

    # ================================================================
    # I. CODE-SWITCHING (campur Bahasa Indonesia + Inggris)
    # ================================================================
    {"kategori": "I-code_switch", "catatan": "campur Inggris di beberapa klausa",
     "teks": "overall nice experience, pemandangannya amazing banget apalagi pas sunrise, "
             "cuma toilet facilitynya masih perlu improvement, staffnya so far friendly semua."},

    {"kategori": "I-code_switch", "catatan": "campur Inggris + saran perbaikan",
     "teks": "the view is stunning tapi jalan ke lokasinya lumayan effort, worth it kok kalo "
             "emang niat hiking dikit."},

    # ================================================================
    # J. SUB-LOKASI SPESIFIK (nama tempat detail dalam kawasan Maribaya)
    # ================================================================
    {"kategori": "J-sub_lokasi", "catatan": "3 curug dibandingkan satu sama lain",
     "teks": "curug cikawari nya paling bagus dibanding 2 curug lain disitu, airnya lebih "
             "jernih, cuma jalan kesana agak jauh dan menurun terus."},

    {"kategori": "J-sub_lokasi", "catatan": "glamping tent + AC, keluhan spesifik",
     "teks": "nginep di glamping tentnya nyaman banget, kasurnya empuk, cuma AC nya kurang "
             "dingin pas malem jadi agak gerah."},

    {"kategori": "J-sub_lokasi", "catatan": "wahana spesifik (balon udara) + antrean",
     "teks": "naik balon udaranya worth it banget buat foto-foto, meskipun antrenya lumayan "
             "panjang pas siang."},

    # ================================================================
    # K. SARAN PERBAIKAN EKSPLISIT (actionable, bukan sekadar venting)
    # ================================================================
    {"kategori": "K-saran_perbaikan", "catatan": "2 saran konkret berbeda topik",
     "teks": "sebaiknya ditambah lagi tempat sampahnya soalnya masih banyak yang buang "
             "sembarangan, terus rambu penunjuk arah ke curug juga kurang jelas jadi sempet "
             "nyasar."},

    {"kategori": "K-saran_perbaikan", "catatan": "kritik ke petugas dengan saran spesifik",
     "teks": "petugas parkirnya harusnya lebih tegas ngatur biar gak semrawut, soalnya kemarin "
             "motor numpuk gak beraturan susah keluar masuk."},

    # ================================================================
    # L. SANGAT PANJANG, MULTI-PARAGRAF (uji batas performa di teks panjang)
    # ================================================================
    {"kategori": "L-sangat_panjang", "catatan": "gabungan hampir semua pola di atas dalam 1 ulasan",
     "teks": "jadi weekend kemarin akhirnya kesampean juga kesini setelah lama pengen. "
             "sampe lokasi jam 7 pagi parkirannya masih longgar untung banget, harga parkir "
             "motor 5rb wajar sih. masuk ke area kolam air panas duluan, airnya anget pas gak "
             "kepanasan, bau belerangnya juga gak terlalu menyengat kayak yang aku takutin "
             "sebelumnya. abis itu jalan ke arah curug, jalannya menurun dan agak licin jadi "
             "hati-hati banget apalagi bawa anak kecil. curugnya sendiri bagus banget airnya "
             "jernih, tapi sayang banyak sampah plastik ngambang padahal udah ada himbauan "
             "jangan buang sampah sembarangan. lanjut ke area sky bridge di the lodge, disini "
             "yang paling rame, antrenya bisa sampe 30 menit lebih, untung petugasnya cukup "
             "sigap ngatur antrean jadi gak terlalu chaos. buat makan siang mampir warung "
             "sekitar situ, menunya lumayan variatif, harganya standar tempat wisata lah gak "
             "terlalu mencekik. yang agak disayangkan toiletnya kurang banyak jadi antre juga, "
             "dan mushola nya kecil banget buat kapasitas pengunjung sebanyak itu. overall "
             "worth it sih buat healing, cuma kalo bisa toilet dan mushola nya diperbanyak "
             "bakal jauh lebih nyaman lagi."},

    # ================================================================
    # M. GENERIK PENDEK (kontrol/baseline -- sengaja HANYA 3, sesuai arahan
    #    agar tidak mendominasi set uji)
    # ================================================================
    {"kategori": "M-generik_pendek", "catatan": "baseline: pujian generik tanpa aspek",
     "teks": "recommended banget pokoknya!"},

    {"kategori": "M-generik_pendek", "catatan": "baseline: netral generik",
     "teks": "lumayan lah buat healing weekend."},

    {"kategori": "M-generik_pendek", "catatan": "baseline: negasi generik yang TIDAK boleh "
                                                  "dipaksa nempel ke aspek manapun",
     "teks": "gak ada komplain sih so far, semua lancar dari masuk sampe pulang."},
]

for item in MARIBAYA_TEST_REVIEWS:
    print("="*70)
    print(f"[{item['kategori']}] {item['catatan']}")
    print("teks:", item['teks'])

    # PERBAIKAN BUG: predict_one() TIDAK PERNAH return None -- dia raise RuntimeError
    # kalau gagal (lihat definisinya di Bagian 3). Cek "if hasil is None" yang lama
    # tidak akan pernah True, dan tanpa try/except di sini, satu kalimat gagal saja
    # akan menghentikan seluruh loop 30 kasus uji ini. Ditangkap di sini supaya kasus
    # lain tetap lanjut diuji walau satu kasus gagal.
    try:
        hasil = extractor.predict_one(item['teks'])
    except RuntimeError as e:
        print("  (gagal ekstraksi)", e)
        continue

    for a in hasil.aspects:
        print(f"  {a.aspect.text} (conf={a.aspect.confidence}, kategori={a.category}/{a.category_confidence}): "
              + ", ".join(f"{o.text}({o.confidence})" for o in a.opinions))
    if hasil.implicit_opinions:
        print("  [opini tanpa aspek eksplisit]: "
              + ", ".join(f"{o.text}({o.confidence}, sentimen={o.category})" for o in hasil.implicit_opinions))

[A-multi_aspek] 5 aspek berbeda, tanda baca minim antar klausa
teks: kolam air panasnya enak banget buat rendam kaki, tapi jalan menuju curugnya lumayan jauh dan licin pas hujan, mushola nya kecil banget jadi harus antre lama, parkiran motor juga sempit banget apalagi pas weekend rame.
  kolam air panas (conf=0.984, kategori=FASILITAS/0.978): enak banget(0.99)
  jalan (conf=0.721, kategori=FASILITAS/0.66): lumayan jauh(0.99), licin(0.994)
  mushola (conf=0.961, kategori=FASILITAS/0.939): kecil banget(0.968), lama(0.79)
  parkiran (conf=0.997, kategori=FASILITAS/0.984): sempit banget(0.975)
[A-multi_aspek] harga + pemandangan + kebersihan + warung, 1 paragraf
teks: harga tiket masuk lumayan mahal menurut aku buat fasilitas yang ada, tapi pemandangan hutan pinusnya emang juara, spot fotonya banyak banget, cuma sayang tempat sampah kurang jadi agak kotor di beberapa titik, warung makannya juga lumayan harganya standar kok gak semahal yang aku kira.
  harga (conf=0.986, kategori=HARGA/0.98